In [1]:
# #  ___________________________________________________________________________
# #
# #  Pyomo: Python Optimization Modeling Objects
# #  Copyright (c) 2008-2025
# #  National Technology and Engineering Solutions of Sandia, LLC
# #  Under the terms of Contract DE-NA0003525 with National Technology and
# #  Engineering Solutions of Sandia, LLC, the U.S. Government retains certain
# #  rights in this software.
# #  This software is distributed under the 3-clause BSD License.
# #  ___________________________________________________________________________
# # === Required imports ===
# import pyomo.environ as pyo
# from pyomo.dae import ContinuousSet, DerivativeVar, Simulator

# from pyomo.contrib.parmest.experiment import Experiment


# # ========================
# class ReactorExperiment(Experiment):
#     def __init__(self, data, nfe, ncp):
#         """
#         Arguments
#         ---------
#         data: object containing vital experimental information
#         nfe: number of finite elements
#         ncp: number of collocation points for the finite elements
#         """
#         self.data = data
#         self.nfe = nfe
#         self.ncp = ncp
#         self.model = None

#         #############################
#         # End constructor definition

#     def get_labeled_model(self):
#         if self.model is None:
#             self.create_model()
#             self.finalize_model()
#             self.label_experiment()
#         return self.model

#     # Create flexible model without data
#     def create_model(self):
#         """
#         This is an example user model provided to DoE library.
#         It is a dynamic problem solved by Pyomo.DAE.

#         Return
#         ------
#         m: a Pyomo.DAE model
#         """

#         m = self.model = pyo.ConcreteModel()

#         # Model parameters
#         m.R = pyo.Param(mutable=False, initialize=8.314)

#         # Define model variables
#         ########################
#         # time
#         m.t = ContinuousSet(bounds=[0, 1])

#         # Concentrations
#         m.CA = pyo.Var(m.t, within=pyo.NonNegativeReals)
#         m.CB = pyo.Var(m.t, within=pyo.NonNegativeReals)
#         m.CC = pyo.Var(m.t, within=pyo.NonNegativeReals)

#         # Temperature
#         m.T = pyo.Var(m.t, within=pyo.NonNegativeReals)

#         # Arrhenius rate law equations
#         m.A1 = pyo.Var(within=pyo.NonNegativeReals)
#         m.E1 = pyo.Var(within=pyo.NonNegativeReals)
#         m.A2 = pyo.Var(within=pyo.NonNegativeReals)
#         m.E2 = pyo.Var(within=pyo.NonNegativeReals)

#         # Differential variables (Conc.)
#         m.dCAdt = DerivativeVar(m.CA, wrt=m.t)
#         m.dCBdt = DerivativeVar(m.CB, wrt=m.t)

#         ########################
#         # End variable def.

#         # Equation definition
#         ########################

#         # Expression for rate constants
#         @m.Expression(m.t)
#         def k1(m, t):
#             return m.A1 * pyo.exp(-m.E1 * 1000 / (m.R * m.T[t]))

#         @m.Expression(m.t)
#         def k2(m, t):
#             return m.A2 * pyo.exp(-m.E2 * 1000 / (m.R * m.T[t]))

#         # Concentration odes
#         @m.Constraint(m.t)
#         def CA_rxn_ode(m, t):
#             return m.dCAdt[t] == -m.k1[t] * m.CA[t]

#         @m.Constraint(m.t)
#         def CB_rxn_ode(m, t):
#             return m.dCBdt[t] == m.k1[t] * m.CA[t] - m.k2[t] * m.CB[t]

#         # algebraic balance for concentration of C
#         # Valid because the reaction system (A --> B --> C) is equimolar
#         @m.Constraint(m.t)
#         def CC_balance(m, t):
#             return m.CA[0] == m.CA[t] + m.CB[t] + m.CC[t]

#         ########################
#         # End equation definition

#     def finalize_model(self):
#         """
#         Example finalize model function. There are two main tasks
#         here:

#             1. Extracting useful information for the model to align
#                with the experiment. (Here: CA0, t_final, t_control)
#             2. Discretizing the model subject to this information.

#         """
#         m = self.model

#         # Unpacking data before simulation
#         control_points = self.data["control_points"]

#         # Set initial concentration values for the experiment
#         m.CA[0].value = self.data["CA0"]
#         m.CB[0].fix(self.data["CB0"])

#         # Update model time `t` with time range and control time points
#         m.t.update(self.data["t_range"])
#         m.t.update(control_points)

#         # Fix the unknown parameter values
#         m.A1.fix(self.data["A1"])
#         m.A2.fix(self.data["A2"])
#         m.E1.fix(self.data["E1"])
#         m.E2.fix(self.data["E2"])

#         # Add upper and lower bounds to the design variable, CA[0]
#         m.CA[0].setlb(self.data["CA_bounds"][0])
#         m.CA[0].setub(self.data["CA_bounds"][1])

#         m.t_control = control_points

#         # Discretizing the model
#         discr = pyo.TransformationFactory("dae.collocation")
#         discr.apply_to(m, nfe=self.nfe, ncp=self.ncp, wrt=m.t)

#         # Initializing Temperature in the model
#         cv = None
#         for t in m.t:
#             if t in control_points:
#                 cv = control_points[t]
#                 m.T[t].fix()
#             m.T[t].setlb(self.data["T_bounds"][0])
#             m.T[t].setub(self.data["T_bounds"][1])
#             m.T[t] = cv

#         # Make a constraint that holds temperature constant between control time points
#         @m.Constraint(m.t - control_points)
#         def T_control(m, t):
#             """
#             Piecewise constant temperature between control points
#             """
#             neighbour_t = max(tc for tc in control_points if tc < t)
#             return m.T[t] == m.T[neighbour_t]

#         #########################
#         # End model finalization

#     def label_experiment(self):
#         """
#         Example for annotating (labeling) the model with a
#         full experiment.
#         """
#         m = self.model

#         # Set measurement labels
#         m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
#         # Add CA to experiment outputs
#         m.experiment_outputs.update((m.CA[t], None) for t in m.t_control)
#         # Add CB to experiment outputs
#         m.experiment_outputs.update((m.CB[t], None) for t in m.t_control)
#         # Add CC to experiment outputs
#         m.experiment_outputs.update((m.CC[t], None) for t in m.t_control)

#         # Adding error for measurement values (assuming no covariance and constant error for all measurements)
#         m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
#         concentration_error = 1e-2  # Error in concentration measurement
#         # Add measurement error for CA
#         m.measurement_error.update((m.CA[t], concentration_error) for t in m.t_control)
#         # Add measurement error for CB
#         m.measurement_error.update((m.CB[t], concentration_error) for t in m.t_control)
#         # Add measurement error for CC
#         m.measurement_error.update((m.CC[t], concentration_error) for t in m.t_control)

#         # Identify design variables (experiment inputs) for the model
#         m.experiment_inputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
#         # Add experimental input label for initial concentration
#         m.experiment_inputs[m.CA[m.t.first()]] = None
#         # Add experimental input label for Temperature
#         m.experiment_inputs.update((m.T[t], None) for t in m.t_control)

#         # Add unknown parameter labels
#         m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
#         # Add labels to all unknown parameters with nominal value as the value
#         m.unknown_parameters.update((k, pyo.value(k)) for k in [m.A1, m.A2, m.E1, m.E2])

#         #########################
#         # End model labeling

In [2]:
# #  ___________________________________________________________________________
# #
# #  Pyomo: Python Optimization Modeling Objects
# #  Copyright (c) 2008-2025
# #  National Technology and Engineering Solutions of Sandia, LLC
# #  Under the terms of Contract DE-NA0003525 with National Technology and
# #  Engineering Solutions of Sandia, LLC, the U.S. Government retains certain
# #  rights in this software.
# #  This software is distributed under the 3-clause BSD License.
# #  ___________________________________________________________________________
# from pyomo.common.dependencies import numpy as np, pathlib

# from pyomo.contrib.doe.examples.reactor_experiment import ReactorExperiment
# from pyomo.contrib.doe import DesignOfExperiments

# import pyomo.environ as pyo
# from pyomo.environ import (
#     ConcreteModel,
#     Var,
#     Param,
#     Constraint,
#     TransformationFactory,
#     SolverFactory,
#     Objective,
#     minimize,
#     value as pyovalue,
#     Suffix,
#     Expression,
#     sin,
#     NonNegativeReals,
# )



# data_ex = {"CA0": 5.0, "CA_bounds": [1.0, 5.0], "CB0": 0.0, "CC0": 0.0, "t_range": [0, 1],
#            "control_points": {"0": 500, "0.125": 300, "0.25": 300, "0.375": 300, "0.5": 300, "0.625": 300, "0.75": 300,
#                               "0.875": 300, "1": 300}, "T_bounds": [300, 700], "A1": 84.79, "A2": 371.72, "E1": 7.78, "E2": 15.05}
# # Put temperature control time points into correct format for reactor experiment
# data_ex["control_points"] = {
#     float(k): v for k, v in data_ex["control_points"].items()
# }

# # Create a ReactorExperiment object; data and discretization information are part
# # of the constructor of this object
# experiment = ReactorExperiment(data=data_ex, nfe=10, ncp=3)

# # Use a central difference, with step size 1e-3
# fd_formula = "central"
# step_size = 1e-3

# # Use the determinant objective with scaled sensitivity matrix
# objective_option = "determinant"
# scale_nominal_param_value = True

# # Create the DesignOfExperiments object
# # We will not be passing any prior information in this example.
# # We also will rely on the initialization routine within
# # the DesignOfExperiments class.
# doe_obj = DesignOfExperiments(
#     experiment,
#     fd_formula=fd_formula,
#     step=step_size,
#     objective_option=objective_option,
#     gradient_method = "pynumero",
#     scale_constant_value=1,
#     scale_nominal_param_value=scale_nominal_param_value,
#     prior_FIM=None,
#     jac_initial=None,
#     fim_initial=None,
#     L_diagonal_lower_bound=1e-7,
#     solver=SolverFactory('IPOPT'),
#     tee=False,
#     get_labeled_model_args=None,
#     _Cholesky_option=True,
#     _only_compute_fim_lower=True,
# )

# # Make design ranges to compute the full factorial design
# design_ranges = {"CA[0]": [1, 5, 9], "T[0]": [300, 700, 9]}

# # # Compute the full factorial design with the sequential FIM calculation
# # doe_obj.compute_FIM_full_factorial(design_ranges=design_ranges, method="sequential")

# # # Plot the results
# # doe_obj.draw_factorial_figure(
# #     sensitivity_design_variables=["CA[0]", "T[0]"],
# #     fixed_design_variables={
# #         "T[0.125]": 300,
# #         "T[0.25]": 300,
# #         "T[0.375]": 300,
# #         "T[0.5]": 300,
# #         "T[0.625]": 300,
# #         "T[0.75]": 300,
# #         "T[0.875]": 300,
# #         "T[1]": 300,
# #     },
# #     title_text="Reactor Example",
# #     xlabel_text="Concentration of A (M)",
# #     ylabel_text="Initial Temperature (K)",
# #     figure_file_name="example_reactor_compute_FIM",
# #     log_scale=False,
# # )


# doe_obj.run_doe()

# # Print out a results summary
# print("Optimal experiment values: ")
# print(
#     "\tInitial concentration: {:.2f}".format(
#         doe_obj.results["Experiment Design"][0]
#     )
# )
# print(
#     ("\tTemperature values: [" + "{:.2f}, " * 8 + "{:.2f}]").format(
#         *doe_obj.results["Experiment Design"][1:]
#     )
# )
# print("FIM at optimal design:\n {}".format(np.array(doe_obj.results["FIM"])))
# print(
#     "Objective value at optimal design: {:.2f}".format(
#         pyo.value(doe_obj.model.objective)
#     )
# )

# print(doe_obj.results["Experiment Design Names"])

# print(sorted(doe_obj.results.keys()))

# print("Solve time (s):", doe_obj.results["Solve Time"])
# print("Build time (s):", doe_obj.results["Build Time"])
# print("Initialization time (s):", doe_obj.results["Initialization Time"])
# print("Total wall time (s):", doe_obj.results["Wall-clock Time"])

# ###################
# # End optimal DoE



In [3]:
# from metrics_logger import log_benchmark

# NOTEBOOK_ID = "4"
# scenario = "symbolic"
# log_benchmark(
#     notebook_id = NOTEBOOK_ID,
#     scenario = scenario,
#     solve_time_s = doe_obj.results["Solve Time"],
#     build_time_s = doe_obj.results["Build Time"],
#     init_time_s = doe_obj.results["Initialization Time"],
#     total_wall_time_s = doe_obj.results["Wall-clock Time"]
# )

In [4]:
    #  ___________________________________________________________________________
#
#  Pyomo: Python Optimization Modeling Objects
#  Copyright (c) 2008-2025
#  National Technology and Engineering Solutions of Sandia, LLC
#  Under the terms of Contract DE-NA0003525 with National Technology and
#  Engineering Solutions of Sandia, LLC, the U.S. Government retains certain
#  rights in this software.
#  This software is distributed under the 3-clause BSD License.
#  ___________________________________________________________________________

# === Required imports ===
from pathlib import Path
import shutil

import pyomo.environ as pyo
from pyomo.dae import ContinuousSet, DerivativeVar
from pyomo.contrib.parmest.experiment import Experiment
from pyomo.common.dependencies import numpy as np
from pyomo.contrib.doe import DesignOfExperiments

# --------------------
# IPOPT: explicit setup (prevents "Solver (IPOPT) not available" on CRC)
# --------------------
IPOPT_BIN = shutil.which("ipopt")
assert IPOPT_BIN, "ipopt executable not found on PATH (activate the 'doe' env)."
ipopt_solver = pyo.SolverFactory("ipopt", executable=IPOPT_BIN)

# --------------------
# Excel helpers (one workbook, one sheet)
# --------------------
try:
    from openpyxl import Workbook, load_workbook
except ImportError as e:
    raise ImportError("openpyxl is required. Install with: conda install -c conda-forge openpyxl") from e

def _ensure_book_and_sheet(xlsx_path: Path, sheet_name: str,
                           header_cols: list[str], notebook_id: str, scenario: str):
    """
    Ensure outputs/metrics.xlsx exists and has a sheet `sheet_name`.
    On first creation of that sheet, write NOTEBOOK_ID & scenario once (row 1),
    then a blank row, then the header row.
    """
    xlsx_path.parent.mkdir(parents=True, exist_ok=True)
    if xlsx_path.exists():
        wb = load_workbook(xlsx_path)
    else:
        wb = Workbook()
        if wb.sheetnames == ["Sheet"]:
            wb.remove(wb["Sheet"])

    if sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        if ws.max_row == 1 and all(c.value is None for c in ws[1]):
            ws.append(["NOTEBOOK_ID", notebook_id, "SCENARIO", scenario])
            ws.append([])
            ws.append(header_cols)
    else:
        ws = wb.create_sheet(title=sheet_name[:31])  # Excel sheet name limit
        ws.append(["NOTEBOOK_ID", notebook_id, "SCENARIO", scenario])
        ws.append([])
        ws.append(header_cols)

    return wb, ws

def _append_row(ws, row):
    ws.append(row)

def _save_wb(wb, xlsx_path: Path):
    wb.save(xlsx_path)

# ========================
# Experiment definition
# ========================
class ReactorExperiment(Experiment):
    def __init__(self, data, nfe, ncp):
        self.data = data
        self.nfe = nfe
        self.ncp = ncp
        self.model = None

    def get_labeled_model(self):
        if self.model is None:
            self.create_model()
            self.finalize_model()
            self.label_experiment()
        return self.model

    def create_model(self):
        """
        Dynamic problem solved by Pyomo.DAE.
        """
        m = self.model = pyo.ConcreteModel()

        # Model parameters
        m.R = pyo.Param(mutable=False, initialize=8.314)

        # time
        m.t = ContinuousSet(bounds=[0, 1])

        # Concentrations
        m.CA = pyo.Var(m.t, within=pyo.NonNegativeReals)
        m.CB = pyo.Var(m.t, within=pyo.NonNegativeReals)
        m.CC = pyo.Var(m.t, within=pyo.NonNegativeReals)

        # Temperature
        m.T = pyo.Var(m.t, within=pyo.NonNegativeReals)

        # Arrhenius parameters
        m.A1 = pyo.Var(within=pyo.NonNegativeReals)
        m.E1 = pyo.Var(within=pyo.NonNegativeReals)
        m.A2 = pyo.Var(within=pyo.NonNegativeReals)
        m.E2 = pyo.Var(within=pyo.NonNegativeReals)

        # Differential variables
        m.dCAdt = DerivativeVar(m.CA, wrt=m.t)
        m.dCBdt = DerivativeVar(m.CB, wrt=m.t)

        # k(T)
        @m.Expression(m.t)
        def k1(m, t):
            return m.A1 * pyo.exp(-m.E1 * 1000 / (m.R * m.T[t]))

        @m.Expression(m.t)
        def k2(m, t):
            return m.A2 * pyo.exp(-m.E2 * 1000 / (m.R * m.T[t]))

        # ODEs
        @m.Constraint(m.t)
        def CA_rxn_ode(m, t):
            return m.dCAdt[t] == -m.k1[t] * m.CA[t]

        @m.Constraint(m.t)
        def CB_rxn_ode(m, t):
            return m.dCBdt[t] == m.k1[t] * m.CA[t] - m.k2[t] * m.CB[t]

        # Algebraic balance for C (equimolar A->B->C)
        @m.Constraint(m.t)
        def CC_balance(m, t):
            return m.CA[0] == m.CA[t] + m.CB[t] + m.CC[t]

    def finalize_model(self):
        m = self.model
        control_points = self.data["control_points"]

        # Initial conditions
        m.CA[0].value = self.data["CA0"]
        m.CB[0].fix(self.data["CB0"])

        # Time grid & control points
        m.t.update(self.data["t_range"])
        m.t.update(control_points)

        # Fix unknown parameters
        m.A1.fix(self.data["A1"])
        m.A2.fix(self.data["A2"])
        m.E1.fix(self.data["E1"])
        m.E2.fix(self.data["E2"])

        # Bounds
        m.CA[0].setlb(self.data["CA_bounds"][0])
        m.CA[0].setub(self.data["CA_bounds"][1])
        m.t_control = control_points

        # Discretize
        pyo.TransformationFactory("dae.collocation").apply_to(m, nfe=self.nfe, ncp=self.ncp, wrt=m.t)

        # Initialize & bound temperature (piecewise-constant between control points)
        cv = None
        for t in m.t:
            if t in control_points:
                cv = control_points[t]
                m.T[t].fix()
            m.T[t].setlb(self.data["T_bounds"][0])
            m.T[t].setub(self.data["T_bounds"][1])
            m.T[t] = cv

        @m.Constraint(m.t - control_points)
        def T_control(m, t):
            neighbour_t = max(tc for tc in control_points if tc < t)
            return m.T[t] == m.T[neighbour_t]

    def label_experiment(self):
        m = self.model
        m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.experiment_outputs.update((m.CA[t], None) for t in m.t_control)
        m.experiment_outputs.update((m.CB[t], None) for t in m.t_control)
        m.experiment_outputs.update((m.CC[t], None) for t in m.t_control)

        m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        concentration_error = 1e-2
        m.measurement_error.update((m.CA[t], concentration_error) for t in m.t_control)
        m.measurement_error.update((m.CB[t], concentration_error) for t in m.t_control)
        m.measurement_error.update((m.CC[t], concentration_error) for t in m.t_control)

        m.experiment_inputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.experiment_inputs[m.CA[m.t.first()]] = None
        m.experiment_inputs.update((m.T[t], None) for t in m.t_control)

        m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.unknown_parameters.update((k, pyo.value(k)) for k in [m.A1, m.A2, m.E1, m.E2])

#  ___________________________________________________________________________

# (avoid importing the example ReactorExperiment to prevent shadowing the class above)
# from pyomo.contrib.doe.examples.reactor_experiment import ReactorExperiment  # <-- intentionally not used

# --------------------
# Static data + options
# --------------------
data_ex = {
    "CA0": 5.0, "CA_bounds": [1.0, 5.0], "CB0": 0.0, "CC0": 0.0, "t_range": [0, 1],
    "control_points": {"0": 500, "0.125": 300, "0.25": 300, "0.375": 300, "0.5": 300,
                       "0.625": 300, "0.75": 300, "0.875": 300, "1": 300},
    "T_bounds": [300, 700], "A1": 84.79, "A2": 371.72, "E1": 7.78, "E2": 15.05
}
data_ex["control_points"] = {float(k): v for k, v in data_ex["control_points"].items()}

fd_formula = "central"
step_size = 1e-3
objective_option = "determinant"
scale_nominal_param_value = True

# Notebook / Scenario / Excel config
NOTEBOOK_ID = "4"
scenario = "symbolic"
XLSX_PATH = Path("outputs/metrics.xlsx")
SHEET_NAME = f"{NOTEBOOK_ID}_{scenario}"[:31]  # Excel sheet name limit

ROW_HEADER = [
    "run", "solve_time_s", "build_time_s", "init_time_s", "total_wall_time_s", "optimal_experiment_values"
]

# Ensure workbook & sheet ready
_wb, _ws = _ensure_book_and_sheet(XLSX_PATH, SHEET_NAME, ROW_HEADER, NOTEBOOK_ID, scenario)
_save_wb(_wb, XLSX_PATH)

# --------------------
# One run = build fresh objects + call run_doe() exactly once
# --------------------
def run_single():
    experiment = ReactorExperiment(data=data_ex, nfe=10, ncp=3)

    doe_obj = DesignOfExperiments(
        experiment,
        fd_formula=fd_formula,
        step=step_size,
        objective_option=objective_option,
        gradient_method="pynumero",          # keep as in your original
        scale_constant_value=1,
        scale_nominal_param_value=scale_nominal_param_value,
        prior_FIM=None,
        jac_initial=None,
        fim_initial=None,
        L_diagonal_lower_bound=1e-7,
        solver=ipopt_solver,                  # explicit IPOPT handle (fixes CRC error)
        tee=False,
        get_labeled_model_args=None,
        _Cholesky_option=True,
        _only_compute_fim_lower=True,
    )

    doe_obj.run_doe()

    # metrics
    solve_time = doe_obj.results["Solve Time"]
    build_time = doe_obj.results["Build Time"]
    init_time  = doe_obj.results["Initialization Time"]
    total_time = doe_obj.results["Wall-clock Time"]

    # optimal experimental values (as compact CSV string)
    design_vals = doe_obj.results["Experiment Design"]
    design_vals_str = ",".join(f"{v:.6g}" for v in design_vals)

    return solve_time, build_time, init_time, total_time, design_vals_str

# --------------------
# Run 100 times, store rows into Excel, print run number each iteration,
# and print the 100th run's data at the end.
# --------------------
wb = load_workbook(XLSX_PATH) if XLSX_PATH.exists() else Workbook()
ws = wb[SHEET_NAME] if SHEET_NAME in wb.sheetnames else wb.create_sheet(title=SHEET_NAME)
if ws.max_row == 1 and all(c.value is None for c in ws[1]):
    ws.append(["NOTEBOOK_ID", NOTEBOOK_ID, "SCENARIO", scenario])
    ws.append([])
    ws.append(ROW_HEADER)

results_100 = None

for run in range(1, 101):
    print(f"Run {run}")
    solve_time, build_time, init_time, total_time, design_vals_str = run_single()
    _append_row(ws, [run, solve_time, build_time, init_time, total_time, design_vals_str])
    if run == 100:
        results_100 = (run, solve_time, build_time, init_time, total_time, design_vals_str)

_save_wb(wb, XLSX_PATH)

# Print only the 100th run's data
if results_100 is not None:
    r, st, bt, it, tt, dv = results_100
    print(f"100th run -> run={r}, solve_time_s={st}, build_time_s={bt}, init_time_s={it}, total_wall_time_s={tt}")
    print(f"Optimal experimental values (run 100): {dv}")


Run 1

Measurement Mapping:
key : scenario_blocks[0].CA[0.125] value: 17
key : scenario_blocks[0].CA[0.25] value: 34
key : scenario_blocks[0].CA[0.375] value: 42
key : scenario_blocks[0].CA[0.5] value: 50
key : scenario_blocks[0].CA[0.625] value: 58
key : scenario_blocks[0].CA[0.75] value: 66
key : scenario_blocks[0].CA[0.875] value: 74
key : scenario_blocks[0].CA[1] value: 82
key : scenario_blocks[0].CB[0.125] value: 95
key : scenario_blocks[0].CB[0.25] value: 107
key : scenario_blocks[0].CB[0.375] value: 113
key : scenario_blocks[0].CB[0.5] value: 119
key : scenario_blocks[0].CB[0.625] value: 125
key : scenario_blocks[0].CB[0.75] value: 131
key : scenario_blocks[0].CB[0.875] value: 137
key : scenario_blocks[0].CB[1] value: 143
key : scenario_blocks[0].CC[0] value: 144
key : scenario_blocks[0].CC[0.125] value: 150
key : scenario_blocks[0].CC[0.25] value: 156
key : scenario_blocks[0].CC[0.375] value: 159
key : scenario_blocks[0].CC[0.5] value: 162
key : scenario_blocks[0].CC[0.625] val

KeyboardInterrupt: 